In [1]:
import boto3
import json
import numpy as np
from pathlib import Path
import sys
import tempfile

sys.path.append("../src")

from embeddings import load_embedding_model
from llm_client import BedrockLLMClient
from rag_pipeline import answer_question_with_rag

#### S3 paths

In [2]:
BUCKET_NAME = "multiomic-vae-literature-rag-123223178042-eu-north-1-an"

EMBEDDINGS_KEY = "embeddings/chunk_embeddings.npy"
METADATA_KEY = "embeddings/chunk_metadata.jsonl"

s3 = boto3.client("s3")

#### Load embeddings and metadata

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    embeddings_path = Path(tmpdir) / "chunk_embeddings.npy"
    metadata_path = Path(tmpdir) / "chunk_metadata.jsonl"

    s3.download_file(BUCKET_NAME, EMBEDDINGS_KEY, str(embeddings_path))
    s3.download_file(BUCKET_NAME, METADATA_KEY, str(metadata_path))

    chunk_embeddings = np.load(embeddings_path)

    chunk_metadata = [
        json.loads(line)
        for line in metadata_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

print("Embeddings shape:", chunk_embeddings.shape)
print("Metadata count:", len(chunk_metadata))

Embeddings shape: (660, 384)
Metadata count: 660


In [4]:
#### Load models

In [5]:
embedding_model = load_embedding_model(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

llm_client = BedrockLLMClient(
    model_id="eu.amazon.nova-pro-v1:0",
    region_name="eu-north-1",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Ask Question

In [6]:
question = "What is the main idea of BindVAE?"

result = answer_question_with_rag(
    question=question,
    embedding_model=embedding_model,
    llm_client=llm_client,
    chunk_embeddings=chunk_embeddings,
    chunk_metadata=chunk_metadata,
    top_k=5,
)

print(result["answer"])

The main idea of BindVAE is to use Dirichlet variational autoencoders for de novo motif discovery from accessible chromatin regions. It disentangles input DNA sequences into distinct latent factors that encode cell-type specific in vivo binding signals for individual transcription factors (TFs), composite patterns for TFs involved in cooperative binding, and genomic context surrounding the binding sites. (BindVAE)


#### Show sources

In [7]:
for chunk in result["retrieved_chunks"]:
    print("=" * 80)
    print("Rank:", chunk["rank"])
    print("Score:", chunk["score"])
    print("Paper:", chunk["paper_name"])
    print("Chunk ID:", chunk["chunk_id"])

Rank: 1
Score: 0.2902735769748688
Paper: BindVAE
Chunk ID: 0
Rank: 2
Score: 0.2813146114349365
Paper: BindVAE
Chunk ID: 9
Rank: 3
Score: 0.2627418041229248
Paper: CASTLE
Chunk ID: 15
Rank: 4
Score: 0.2340094894170761
Paper: BindVAE
Chunk ID: 4
Rank: 5
Score: 0.1997372806072235
Paper: BindVAE
Chunk ID: 13
